## 第6章 面向对象编程、装饰器和迭代器

### 1.装饰器

- 装饰器是一种函数，接受一个函数作为输入并会返回另一个函数。装饰器的目的是在不改变原函数代码的情况下，为原函数添加一些额外的功能。通常用于日志记录、权限检查、性能测试等场景。
    - 可以使用`@`符号来应用装饰器。
    - 可以使用`@wraps`装饰器来保留原函数的元数据(如函数名、文档字符串等)。
    - 一个函数可应用多个装饰器，**离函数定义最近的装饰器先执行**。
    - 装饰器可以接受参数，一般用于生成动态的装饰器(称为装饰器工厂)。

In [ ]:
from time import sleep,time
from functools import wraps

# 定义一个装饰器，用于测量函数的执行时间
def timer(func):
    @wraps(func)  # 保留原始函数的元数据
    def wrapper(*args, **kwargs):
        start = time()
        sleep(0.15)  # 模拟函数执行时间
        result = func(*args, **kwargs)
        end = time()
        print(f"{func.__name__}执行时间:{end - start}")
        return result
    return wrapper

# 定义一个装饰器，用于检查函数的返回值是否超过100
def maxer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        if result > 100: print(f"计算结果{result}超过100")
        return result
    return wrapper

@timer
@maxer  # 先装饰maxer，再装饰timer
def cube(x):
    return x ** 3

print(cube(2))  # 输出：cube执行时间:0.15... → 8
print(cube(5))  # 输出：计算结果125超过100 → cube执行时间:0.15... → 125

### 2.面向对象编程

- 对象是一种数据结构，由属性(数据)和方法(函数)组成。
- 类是**模板**，用于创建对象。对象是类的**实例**。Python中，类通过`class`语句来定义。
    - 语法：
        ```python
        class 类名(父类1, 父类2, ...):
            类属性 = 值
            ...

            def __init__(self, 参数, ...):
                super().__init__()
                self.实例属性 = 参数
                ...
            
            实例方法(self):
                方法体

            @classmethod
            类方法(cls, 参数, ...):
                方法体

            @staticmethod
            静态方法(参数, ...):
                方法体
        ```
    - 说明：
        - 类属性：定义在类本身上的变量，直接写在类的主体中，**为所有实例共享**。
        - 实例属性：通常通过`self.属性名`在`__init__`方法(或其他实例方法)中定义，**只属于该实例**，不会影响其他实例的属性值。
            - 实例查找属性时，先在自身命名空间查找，未找到则向上查找类的命名空间。若实例赋值了与类属性同名的属性，会临时遮蔽类属性；删除实例属性后，查找会回退到类属性。
        - self：指向当前实例的引用，用于访问实例的属性和方法。约定为实例方法的第一个参数。
        - \_\_init\_\_：初始化方法，在对象创建后自动执行，用于设置实例属性，也可以调用`super().__init__()`初始化父类属性。它不是构造函数，真正的构造函数是`__new__`。

In [ ]:
class Point:
    x = 7           # 类属性
    y = 8           # 类属性

p = Point()         # 创建一个实例对象
print(p.x)          # 输出：7，实例属性x未定义，所以使用类属性x
print(p.y)          # 输出：8，实例属性y未定义，所以使用类属性y

p.x = 10            # 定义实例属性x
print(p.x)          # 输出：10，使用实例属性x
print(Point.x)      # 输出：7，显示声明使用类属性x

del p.x             # 删除实例属性x
print(p.x)          # 输出：7，实例属性x已删除，所以使用类属性x

- 合成：两个对象通过一种“Has-A”(有)类型的关系进行关联。对象由其他对象组成。
- 继承：两个对象通过一种“Is-A”(是)类型的关系进行关联，子类继承父类的属性和方法。可通过`isinstance()`判断实例类型，`issubclass()`判断子类关系。
    - 语法：`class 子类名(父类名1, 父类名2, ...):`。
    - 访问父类的属性和方法：`super().属性名`或`super().方法名()`。
    - 多重继承：通过`MRO`算法来确定方法的调用顺序，可以通过`类名.__mro__`或`类名.mro()`来查看MRO顺序。
- **设计原则**：优先使用组合，而非继承。继承适合明确的层级关系，组合适合灵活组装。

In [ ]:
# 定义基类
class Engine:
    def start(self): pass
    def stop(self): pass

# 继承(Is-A)
class ElectricEngine(Engine): pass
class V8Engine(Engine): pass

# 定义基类(将Engine类组合到Car类中)
class Car:
    engine_cls = Engine  # 组合(Has-A)
    def __init__(self):
        self.engine = self.engine_cls()
    def start(self):
        print(f"Start engine:{self.engine.__class__.__name__} for car:{self.__class__.__name__}")
        self.engine.start()
    def stop(self):
        self.engine.stop()

# 继承+组合
class RaceCar(Car): engine_cls = V8Engine
class CityCar(Car): engine_cls = ElectricEngine
class F1Car(RaceCar): pass

cars = [Car(), RaceCar(), CityCar(), F1Car()]
for car in cars: car.start()

- 静态方法：在类中定义的，不需要实例化对象即可调用的方法。
    - 语法：`@staticmethod`
    - 作用：通常作为函数容器来组织相关的方法(工具类)，与模块中定义函数类似。
- 类方法：与静态方法略有不同，它和实例方法一样接受一个特殊的第一参数`cls`，用于表示类对象本身而不是实例。
    - 语法：`@classmethod`
    - 作用：类方法的一个非常常见的用途是为类提供工厂功能。提供多种创建实例的方式。

In [ ]:
# 通过静态方法实现工具类
class StringUtils:
    @staticmethod       # 定义静态方法
    def get_unique_words(sentence):
        return set(sentence.split())

# 通过类方法实现工厂类
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    @classmethod        # 定义类方法
    def from_tuple(cls, coords):
        return cls(*coords)

print(StringUtils.get_unique_words("I love palindromes. I really really love them!"))
p = Point.from_tuple((1, 2))
print(p.x, p.y)

- 私有属性和方法：在Python中，所有属性和方法都是**公有的**。为了保护类的内部状态，依靠**约定**和**名称改写**来实现访问控制。
    - 约定：在属性或方法名前添加下划线`_`，这种约定告诉其他开发人员，这个属性或方法是私有的，不应该被外部访问。
    - 名称改写：在属性或方法名前添加双下划线`__`，Python会自动将属性或方法的名称改写为`_类名__属性名`或`_类名__方法名`，外部不能直接访问到私有属性或方法。

- 特性(property装饰器)：当一个方法被装饰为特性时(`@property`、`@属性名.setter`)，它就可以像属性一样被访问和赋值。
    - 在Python3.8+中，还可以使用`@cache_property`装饰器来实现缓存属性，即在第一次访问时计算属性值，后续访问直接直接返回缓存值。

- 数据类(dataclass)：在Python3.7+中，可以使用`@dataclass`装饰器来创建一个数据类，类似于具有默认值的可变命名元组。

In [ ]:
from dataclasses import dataclass

@dataclass
class Car:
    year: int = 2020
    color: str = "Black"
    price: float = 20.0

car = Car(2026, "White", 30.0)
print(car)      # Car(year=2026, color='White', price=30.0)

### 3.迭代器

- 可迭代对象：可一次返回一个成员的对象，也即定义了`__iter__()`或`__getitem__()`方法的对象，列表、元组、字符串、字典都是可迭代对象。
- 迭代器：代表数据流的临时对象，必须实现两个方法：`__iter__`(返回自身)、`__next__`(返回下一个元素，数据流耗尽时抛出StopIteration异常)。内置`iter()`和`next()`函数分别对应调用这两个方法。

In [ ]:
# 创建奇偶迭代器
class OddEven:
    def __init__(self, data):
        self._data = data
        self.indexes = (list(range(0, len(data), 2)) + list(range(1, len(data), 2)))  # 分开奇偶索引

    # 实现可迭代对象
    def __iter__(self):
        return self

    # 实现迭代器
    def __next__(self):
        if self.indexes:
            return self._data[self.indexes.pop(0)]  # 返回下一个元素
        else:
            raise StopIteration  # 没有更多元素时抛出异常

odd = OddEven('1234567890')
print(''.join(i for i in odd))  # 输出：1357924680

### 4.本章小结

#### (1)核心内容

```text
OOP、装饰器与迭代器
│
├── 装饰器 ⭐
│   ├── 动机：消除重复代码
│   ├── 基本装饰器：func = decorator(func)
│   ├── @ 语法糖：@decorator
│   ├── 多装饰器：靠近函数的先装饰（由内到外）
│   ├── 带参数装饰器（装饰器工厂）：三层嵌套
│   ├── @wraps：保留原函数元信息 ⭐
│   └── 常见错误：忘记 return result、忘记 @wraps
│
├── OOP ⭐
│   ├── 类与实例（class / __init__ / self）
│   ├── 类属性（共享）vs 实例属性（独有）
│   ├── 三种方法
│   │   ├── 实例方法（self）
│   │   ├── 类方法（@classmethod, cls）
│   │   └── 静态方法（@staticmethod）
│   ├── 继承（Is-a）vs 组合（Has-a）
│   ├── super() 调用父类方法
│   ├── 方法重写（Override）
│   ├── @property 控制属性读写
│   ├── 运算符重载（魔术方法）
│   ├── 多态（隐式/鸭子类型）
│   └── @dataclass（Python 3.7+）
│
└── 自定义迭代器
    ├── Iterable（__iter__ 或 __getitem__）
    └── Iterator（__iter__ + __next__ + StopIteration）
```

#### (2)装饰器模板

**无参数装饰器：**

```python
from functools import wraps

def decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        # 前置逻辑
        result = func(*args, **kwargs)
        # 后置逻辑
        return result
    return wrapper
```

**带参数装饰器：**

```python
def decorator_factory(arg1, arg2):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            # 使用 arg1, arg2
            result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator
```

#### (3)关键警告与提示

| 类型   | 内容                                                         |
| ------ | ------------------------------------------------------------ |
| ⚠️ 警告 | 装饰器中**必须 `return result`**，否则原函数返回值丢失       |
| ⚠️ 警告 | 装饰器中**必须 `@wraps(func)`**，否则原函数的 `__name__`/`__doc__` 丢失 |
| ⚠️ 警告 | 多装饰器顺序：**靠近函数的先装饰**（由内到外）               |
| ⚠️ 警告 | `@property` 名字和内部变量名**不能相同**，否则无限递归       |
| ⚠️ 警告 | 迭代器耗尽后再次 `next()` 抛 `StopIteration`，不能重置       |
| 💡 技巧 | 装饰器工厂 = 三层嵌套：`factory → decorator → wrapper`       |
| 💡 技巧 | 类方法常作**替代构造器**（`@classmethod def from_str(cls, s):`） |
| 💡 技巧 | `@dataclass` 是 `namedtuple` 的可变增强版，适合纯数据+简单方法的类 |
| 💡 技巧 | 自定义迭代器务必测试空序列、长度1、长度2等边界               |